## Assignment 4    UCS420 Cognitive Computing  


1. Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas
DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your
own roll number digits as follows:
Take the LAST TWO DIGITS of your roll number. For each digit d, compute category = ["billing", "account",
"general"][d % 3]. Invent one realistic question+answer+3 keywords per entry that fits the assigned category 

1. Example roll number ...23 -> digits 2, 3
2. digit 2 -> category[2 % 3] = general
3. digit 3 -> category[3 % 3] = billing

In [7]:
# Getting the category
my_roll  = 28
d1=my_roll%10
d2 =my_roll//10

category = ['general','account','billing'] # given
print("For digit 2:",category[d1%3])
print("For digit 8:",category[d2%3])



For digit 2: billing
For digit 8: billing


In [8]:
import pandas as pd

fixed_entries = [
 {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
 "keywords": "fee cost price charge", "category": "billing"},
 {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
 "keywords": "password reset login", "category": "account"},
 {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
 "keywords": "hours timing open time", "category": "general"},
 {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.","keywords": "pay payment upi fee", "category": "billing" },
 {"question": "which banks are allowed for payment?","answer":"SBI,Canara,ICICI Bank and Punjab Bank","keywords":"Banks for payment","category":"billing"},
 {"question": "Is the payment refundable?","answer":"Yes the transcations are refundable","keywords":"refund policy","category":"billing"}
]

df = pd.DataFrame(fixed_entries)
print(df.to_string(index=False))

                            question                                     answer               keywords category
              what is the annual fee                  The annual fee is Rs 500.  fee cost price charge  billing
               how to reset password           Go to Settings > Reset Password.   password reset login  account
         what are your working hours                  We are open 9 AM to 5 PM. hours timing open time  general
               how can i pay the fee You can pay via UPI, card, or net banking.    pay payment upi fee  billing
which banks are allowed for payment?      SBI,Canara,ICICI Bank and Punjab Bank      Banks for payment  billing
          Is the payment refundable?        Yes the transcations are refundable          refund policy  billing


2. Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching
entries ranked by confidence

In [ ]:
def scoring(query,df):
    query_words = set(query.lower().split())
    results=[]
    for index,row in df.iterrows():
        text=str(row['question'])+" "+str(row['keywords'])
        faq_wds = set(text.lower().split()) # set is used to remove duplicate words

        matched_wds=query_words.intersection(faq_wds)

        if(len(query_words)>0):
            score = len(matched_wds)/len(query_words)
        else:
            score = 0 # the query would result in 0 ie empty results list

        if score>0:
            results.append({
                'question':row['question'],
                'answer':row['answer'],
                'category':row['category'],
                'confidence':round(score,2)

            })

    results =sorted(results,key=lambda x:x['confidence'],reverse=True)
    return pd.DataFrame(results)

query = input("Enter your query")
match = scoring(query,df)

if not match.empty:
    print(match.to_string(index=False))
else:
    print("No FAQ found")

                            question                                     answer category  confidence
               how can i pay the fee You can pay via UPI, card, or net banking.  billing         0.5
which banks are allowed for payment?      SBI,Canara,ICICI Bank and Punjab Bank  billing         0.5


3. Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it
using the category of one of the personalized entries from Q1, and print the result.

In [ ]:
def same_category(category_name,df):
    result = []
    for index,row in df.iterrows(): # index gives index of rows and row is the iteration of rows in the dataframe
        if(category_name==row['category']):
            result.append({
                'question':row['question'],
                'category':row['category']
            })
        else:
            continue

    return pd.DataFrame(result)

desired_category = input("enter the category gen/billing/account")
my_result=same_category(desired_category,df)

if not my_result.empty:
    print(my_result.to_string(index=False))

else:
    print("Not categories found!")

                            question category
              what is the annual fee  billing
               how can i pay the fee  billing
which banks are allowed for payment?  billing
          Is the payment refundable?  billing


4. Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and
save your entire updated DataFrame to a CSV file named <your_roll_number>_faq_data.csv

In [15]:
def change_keyword(key,df):
    # i am changing only the first iteration of the keyword
    changed = False
    for index,row in df.iterrows():
        if(key==row['category']):
            print("current keyword is:",row['category'])
            print("Enter your keyword!")
            my_keyword=input("enter your keyword")
            row['category'] = my_keyword
            changed = 1
            print("Saving the data...")
            df.to_csv("1024160028_faq_data.csv")
            return df
            
    
    

key=input("what keyword you want to change (1 only)")
my_data=change_keyword(key,df)
print(my_data.to_string(index=False))

current keyword is: billing
Enter your keyword!
Saving the data...
                            question                                     answer               keywords category
              what is the annual fee                  The annual fee is Rs 500.  fee cost price charge  payment
               how to reset password           Go to Settings > Reset Password.   password reset login  account
         what are your working hours                  We are open 9 AM to 5 PM. hours timing open time  general
               how can i pay the fee You can pay via UPI, card, or net banking.    pay payment upi fee  billing
which banks are allowed for payment?      SBI,Canara,ICICI Bank and Punjab Bank      Banks for payment  billing
          Is the payment refundable?        Yes the transcations are refundable          refund policy  billing


5. Using groupby, print how many FAQ entries you have per category.

In [22]:
print(df.groupby('category')['question'].apply(list))
print(df.groupby('category')['question'].count())

category
account                              [how to reset password]
billing    [how can i pay the fee, which banks are allowe...
general                        [what are your working hours]
payment                             [what is the annual fee]
Name: question, dtype: object
category
account    1
billing    3
general    1
payment    1
Name: question, dtype: int64


6. Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one
— it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that
produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [25]:

def scoring(query,df):
    query_words = set(query.lower().split())
    results=[]
    for index,row in df.iterrows():
        text=str(row['question'])+" "+str(row['keywords'])
        faq_wds = set(text.lower().split()) # set is used to remove duplicate words

        matched_wds=query_words.intersection(faq_wds)

        if(len(query_words)>0):
            score = len(matched_wds)/len(query_words)
        else:
            score = 0 # the query would result in 0 ie empty results list

        if score>0:
            results.append({
                'question':row['question'],
                'answer':row['answer'],
                'category':row['category'],
                'confidence':round(score,2)

            })

    results =sorted(results,key=lambda x:x['confidence'],reverse=True)

    highest_score = results[0]['confidence']
    top_matchs = [row for row in results if row['confidence']==highest_score]
    print("highest score",highest_score)
    print("matches",len(top_matchs))
    
    return pd.DataFrame(top_matchs)

query = input("Enter your query")
match = scoring(query,df) 

if not match.empty:
    print(match.to_string(index=False))
else:
    print("No FAQ found")

#enter fee for tie
#enter pass for not tie


highest score 1.0
matches 1
             question                           answer category  confidence
how to reset password Go to Settings > Reset Password.  account         1.0
